##### Importing libraries

In [1]:
# --------------------------------------------------------------
# ---------------------------------------------------------------

#### Core Libraries
import re
import openai
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, AzureChatOpenAI
import datetime
import time
import os
import numpy as np
import random
from openai import OpenAI
from keybert import KeyBERT
from pydantic import BaseModel, Field
from langchain_openai import OpenAIEmbeddings
import pandas as pd
import spacy
import seaborn as sns
import pickle
from nltk.stem import PorterStemmer
import string
# sklearn for data & TF-IDF features (keyword-based)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
# tensorflow / keras for the neural network
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,models
import spacy
from spacy_langdetect import LanguageDetector
from langdetect import detect
from spacy.language import Language
from transformers import pipeline
import pickle
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


print("All libraries imported successfully!")
print(os.getcwd())
os.chdir(os.getcwd())

All libraries imported successfully!
c:\Users\mbiswas\OneDrive - Capgemini\Documents\MBiswas_2026\Self_Code_2026\GCP_Training_Assignment


In [2]:
##########  Function to remove emojis from text ##########
emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U0001F1F2-\U0001F1F4"  # Macau flag
        u"\U0001F1E6-\U0001F1FF"  # flags
        u"\U0001F600-\U0001F64F"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U0001F1F2"
        u"\U0001F1F4"
        u"\U0001F620"
        u"\u200d"
        u"\u2640-\u2642"
        "]+", flags=re.UNICODE)

In [ ]:
############## Read Data ##############
df_review_eng = pd.read_csv('df_review_eng.csv')
print(df_review_eng.shape)
pattern = r'[\*@#\$&]+'
df_review_eng['review_clean'] = df_review_eng['review_body'].str.replace("[^a-zA-Z#]", " ")
df_review_eng["review_clean"] = df_review_eng.review_body.replace(pattern,'',regex=True)
df_review_eng["review_clean"].replace('', np.nan, inplace=True)
df_review_eng["review_clean_text"] = (df_review_eng["review_clean"].str.replace(emoji_pattern, '', regex=True)
)
df_review_eng=df_review_eng[['review_id','product_name','retailer','review_body','review_rating','review_clean','review_clean_text']]
df_review_eng['str_length']= df_review_eng['review_clean_text'].str.len()
df_review_eng['word_count'] = df_review_eng['review_clean_text'].str.split().str.len()
df_review_eng_filter=df_review_eng[df_review_eng['word_count']>2]

print(df_review_eng_filter.shape[0])

(12478, 12)
11727


In [ ]:
#######getting Key Phrases to Analyse Text#######
def read_reviews_from_excel(df):
    reviews = []
    # df = pd.read_csv(excel_file)
    for index, row in df.iterrows():

        if not pd.isnull(row['review_body']):
            review = {
                'product': row["product_name"],
                'rating':row["review_rating"],
                'retailer': row['retailer'],
                'review_text': row['review_clean_text']
            }
            reviews.append(review)
    return reviews

# function to Extract key_Phrases
def Extract_key_Phrases(text):
  result=[]
  review = text.lower()

  kw_model = KeyBERT()

  key_phrase_model = kw_model.extract_keywords(docs=review, keyphrase_ngram_range=(1,3))
  result = []
  for i in range(1):
    final_keyPhrase = key_phrase_model[i][0]
    result.append(final_keyPhrase)

  return result

# key_phrases = []
# for review in review_data:
#     # print(review)
#     key_words = Extract_key_Phrases(review["review_text"])
#     # print(key_words)
#     key_phrases.append(key_words)

#### Saving key_phrases
# print(len(key_phrases))
# with open("key_phrases.pkl", "wb") as f:
#     pickle.dump(key_phrases, f)


with open("key_phrases.pkl", "rb") as f:
    key_phrases = pickle.load(f)
    
print(len(key_phrases))

key_phrases[0][0]

print(df_review_eng.shape[0])
df_review_eng_copy=df_review_eng.iloc[:len(key_phrases)].copy()
print(len(df_review_eng_copy))
# df_review_eng_copy.head(5)


result=[]
for i in range(len(key_phrases)):
   result.append(key_phrases[i][0])

###### Adding key_phrases to the dataframe #########
# df_review_eng_copy['key_phrase'] = result
# df_review_eng_copy[["review_body","review_clean_text","key_phrase"]].head(8)
df_review_eng_copy=pd.read_csv('df_review_eng_copy.csv')
df_review_eng_copy.head(5)

351
12478
351


,review_body,review_clean_text,key_phrase
0,Such a good price and quality item.,Such a good price and quality item.,good price quality
1,It’s so good 😊,It’s so good,gift absolutely love
2,"Was bought as a gift, they absolutely love it.","Was bought as a gift, they absolutely love it.",add tea coffe
3,I love it you can add it with tea coffe juice ...,I love it you can add it with tea coffe juice ...,diaper changer mat
4,It’s good. It looks just like the picture. The...,It’s good. It looks just like the picture. The...,tiny bottle versus
5,tiny bottle versus expectations,tiny bottle versus expectations,skin probably buy
6,I really wanted to like this product but i fel...,I really wanted to like this product but i fel...,good long chatging
7,Very good but very long chatging time,Very good but very long chatging time,burn shave feel


In [ ]:
### TEST
df_review_eng.head(5)

,review_id,product_name,retailer,review_body,review_rating,review_clean,review_clean_text,str_length,word_count
0,31e5080c18ca20b78795d864f4e2375a,50-Piece Disposable Cotton Earloop 3 Ply Face ...,noon.com,Such a good price and quality item.,5,Such a good price and quality item.,Such a good price and quality item.,35,7
1,0cc53ab5ffd4546e909937962a7f0d2f,"Qunol Ultra CoQ10 100mg, 3x Better Absorption,...",amazon.ae,It’s so good 😊,5,It’s so good 😊,It’s so good,13,3
2,343569483985c47226e4dec7e8974649,"infinitoo Essential Oils Diffuser, 300ml Ultra...",amazon.ae,"Was bought as a gift, they absolutely love it.",5,"Was bought as a gift, they absolutely love it.","Was bought as a gift, they absolutely love it.",46,9
3,141c0c99072c5df7891f57f213a1e43f,Super Collagen Type 1 And 3 Powder 198g,noon.com,I love it you can add it with tea coffe juice ...,5,I love it you can add it with tea coffe juice ...,I love it you can add it with tea coffe juice ...,51,12
4,0d7df5c7516c7eac1d32a041f1356ff1,"Diaper Bag Backpack, LEQUEEN Waterproof Stylis...",amazon.ae,It’s good. It looks just like the picture. The...,5,It’s good. It looks just like the picture. The...,It’s good. It looks just like the picture. The...,281,54


In [32]:
from sklearn.preprocessing import LabelEncoder
import joblib

le = LabelEncoder()

In [45]:
le = LabelEncoder()
y = le.fit_transform(df_review_eng["review_rating"])
n_classes = len(le.classes_)   # expect 6
print("Classes:", le.classes_)
print("Number of classes:", n_classes)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),  # Unigrams and bigrams
    min_df=5,  # Minimum document frequency
    max_features=5000,  # Maximum number of features
    stop_words='english'
)

text=df_review_eng["review_clean_text"].tolist()
print(len(text))
X = vectorizer.fit_transform(text)
print(len(X.toarray()))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# x_train_dense = X_train.toarray()
# x_test_dense = X_test.toarray()
num_classes = len(y)
x_train_dense = X_train.todense()
x_test_dense = X_test.todense()
# num_classes


y_train_enc = le.fit_transform(y_train)  # Fit once on training labels
y_test_enc  = le.transform(y_test)       # Reuse, no fit

print("Encoder classes:", le.classes_)       # should be [0 1 2 3 4 5]
print("Num classes:", len(le.classes_))      # should be 6

joblib.dump(le, "label_encoder.pkl")         # save for inference


Classes: [0 1 2 3 4 5]
Number of classes: 6
12478
12478
Encoder classes: [0 1 2 3 4 5]
Num classes: 6


['label_encoder.pkl']

In [50]:
n_features = x_train_dense.shape[1]  # likely 12478
n_classes  = len(le.classes_)        # 6
print("Number of features:", n_features)
print("Number of classes:", n_classes)

Number of features: 4820
Number of classes: 6


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(x_train_dense.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model1 = models.Sequential([
    layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(n_classes, activation='softmax')
])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model1.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history1 = model1.fit(
    x_train_dense, y_train,   # convert sparse matrix to dense
    validation_data=(x_test_dense, y_test),
    epochs=80,
    callbacks=[early_stop],
    batch_size=25
)

val_loss, val_acc = model1.evaluate(x_test_dense, y_test, verbose=0)
print(f"Validation accuracy: {val_acc:.3f}")



C:\Users\mbiswas\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/80
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6653 - loss: 1.0209 - val_accuracy: 0.7143 - val_loss: 0.8465
Epoch 2/80
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7385 - loss: 0.7265 - val_accuracy: 0.7320 - val_loss: 0.8171
Epoch 3/80
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7955 - loss: 0.5686 - val_accuracy: 0.7268 - val_loss: 0.8871
Epoch 4/80
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8515 - loss: 0.4288 - val_accuracy: 0.7103 - val_loss: 0.9599
Epoch 5/80
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8936 - loss: 0.3175 - val_accuracy: 0.7039 - val_loss: 1.1169
Epoch 6/80
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9169 - loss: 0.2494 - val_accuracy: 0.7155 - val_loss: 1.2703
Epoch 7/80
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9346 - loss: 0.2014 - val_accuracy: 0.7051 - val_loss: 1.3878
Validation accuracy: 0.732


In [ ]:
test_df=df_review_eng_copy.sample(100)
test_df.sample(5)
df_review_eng_copy["review_clean_text"].iloc[44]
new_text=df_review_eng_copy.loc[[6,44,293], "review_clean_text"].values.tolist()
new_text[2]
# len(new_text)


,Unnamed: 0,review_id,retailer,product_name,product_url,review_url,review_timestamp,review_rating,review_body,review_clean,Language,Lang Probability,str_length,key_phrases
319,649,9a279400c3741dca075c6cf1ffd4aa12,amazon.ae,Honest Beauty Extreme Length Mascara + Lash Pr...,https://www.amazon.ae/dp/B07F462B2D,https://www.amazon.ae/product-reviews/B07F462B...,03-02-2021,1,Horrible wand. Short hairs makes the lashes cl...,Horrible wand. Short hairs makes the lashes cl...,en,0.999997,59,wand short hairs
96,194,d80ba4a4a84845188ff1d5e47cbd5b44,noon.com,2-Piece Silicone Face Mask Brush Multicolour,https://www.noon.com/uae-en/product/N19877031A/p,https://www.noon.com/uae-en/product/N19877031A/p,21-05-2021,5,Affordable useful high quality,Affordable useful high quality,en,0.999996,30,useful high quality
303,616,15e76bc4b05580ade6d67298cd4026b8,amazon.ae,"Tobeape® 500ml Aromatherapy Diffuser, Remote C...",https://www.amazon.ae/dp/B086DPLQMZ,https://www.amazon.ae/product-reviews/B086DPLQ...,19-01-2021,5,Swear this is my new Favourite Aroma Diffuser ...,Swear this is my new Favourite Aroma Diffuser ...,en,0.999997,312,aroma diffuser love
231,474,eb13fac1b0712de0f1548a77dd54ea5e,amazon.ae,Tommee Tippee Fresh Food Feeder (Pack of 1),https://www.amazon.ae/dp/B07VMGGXPQ,https://www.amazon.ae/product-reviews/B07VMGGX...,18-01-2021,5,Very nice product for teething phase of baby.....,Very nice product for teething phase of baby.....,en,0.999996,62,baby safe secure
9,18,de53e0cda795aca2893fa83a3e773d13,amazon.ae,Xiaomi 2019 Newest Global Intelligent Body Fat...,https://www.amazon.ae/dp/B07W4DTRTM,https://www.amazon.ae/product-reviews/B07W4DTR...,03-02-2021,5,So Smart and very good price,So Smart and very good price,en,0.999997,28,smart good price


In [37]:
model1.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 128)            │       284,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 878,420 (3.35 MB)

 Trainable params: 292,806 (1.12 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 585,614 (2.23 MB)

In [51]:
model1.layers

[<Dense name=dense_15, built=True>,
 <Dropout name=dropout_8, built=True>,
 <Dense name=dense_16, built=True>,
 <Dense name=dense_17, built=True>]

In [ ]:
X_new = vectorizer.transform(new_text)
X_new_dense = X_new.todense().astype("float32")  
probs = model1.predict(X_new_dense)   
print(len(probs))

print("Classes from encoder:", le.classes_)
print("Number of classes:", len(le.classes_))
print("probs shape:", probs.shape)

preds = probs.argmax(axis=1).astype(int)  
pred_labels = le.inverse_transform(preds) 

for t, label, p in zip(new_text, pred_labels, probs):
    print(f"Text: {t}\nPredicted rating: {label}\nProbabilities: {np.round(p, 3)}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
3
Classes from encoder: [0 1 2 3 4 5]
Number of classes: 6
probs shape: (3, 6)
Text: I really wanted to like this product but i felt it didn’t do anything to my skin so most probably will not buy again.
Predicted rating: 3
Probabilities: [0.027 0.161 0.122 0.264 0.22  0.207]

Text: I received a different item Different name Different smell I am returning it
Predicted rating: 1
Probabilities: [0.    0.976 0.015 0.004 0.002 0.003]

Text: Hi seller. I do agree the that incense create a flow and comes in beautiful colors and that's it. It someone is buying this product please note that the smell is nothing like what is described. They all smell the same, like burned wood with bad smell.
Predicted rating: 1
Probabilities: [0.003 0.855 0.059 0.036 0.02  0.027]



In [54]:
from sklearn.metrics import classification_report

y_val_pred = model.predict(x_test_dense).argmax(axis=1)
# print(classification_report(y_test, y_val_pred, target_names=le.classes_))


X_new = vectorizer.transform(new_text).toarray().astype("float32")
probs = model1.predict(X_new)
preds = probs.argmax(axis=1)
pred_labels = le.inverse_transform(preds)

for t, label in zip(new_texts, pred_labels):
    print(f"Text: {t}\nPredicted class: {label}\n")


78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Text: I
Predicted class: 3

Text:  
Predicted class: 1

Text: r
Predicted class: 1



In [ ]:
# pip install transformers datasets torch

In [66]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(set(y))   # number of classes in your labels
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\AAYAN\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AAYAN\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [67]:
# # Save the model and tokenizer using pickle
with open("model_bert.pkl", "wb") as f:
    pickle.dump(model, f)



In [68]:
# Encode reviews
encodings = tokenizer(text, truncation=True, padding=True, max_length=256)

# Convert to torch tensors
input_ids = torch.tensor(encodings["input_ids"])
attention_masks = torch.tensor(encodings["attention_mask"])
labels = torch.tensor(y)   # make sure y is numeric (LabelEncoder if needed)

from sklearn.model_selection import train_test_split

train_ids, test_ids, train_masks, test_masks, y_train, y_test = train_test_split(
    input_ids, attention_masks, labels, test_size=0.2, stratify=labels
)

train_data = torch.utils.data.TensorDataset(train_ids, train_masks, y_train)
test_data = torch.utils.data.TensorDataset(test_ids, test_masks, y_test)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

for epoch in range(3):  # usually 2–4 epochs are enough
    model.train()
    for batch in train_loader:
        b_ids, b_masks, b_labels = [x.to(device) for x in batch]
        outputs = model(b_ids, attention_mask=b_masks, labels=b_labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

KeyboardInterrupt: 

In [70]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in test_loader:
        b_ids, b_masks, b_labels = [x.to(device) for x in batch]
        outputs = model(b_ids, attention_mask=b_masks)
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == b_labels).sum().item()
        total += b_labels.size(0)

print("Test Accuracy:", correct/total)

KeyboardInterrupt: 